# Preparación del Dataset: MIMIC-III (Cuidados Intensivos)

## 1. Descripción del Dataset
Este caso es muy especial. El conjunto de datos de MIMIC-III se divide en múltiples tablas o archivos separados (`.csv`).

Aquí aplicaremos **Uniones de Tablas Múltiples (Merge)** usando la lógica de bases de datos. En un hospital real, unir las 26 tablas colapsaría cualquier computadora por la cantidad de "Eventos" por segundo. Sin embargo, para tener un dataset súper completo y realista, vamos a unir las **4 tablas más importantes**:
1. `PATIENTS`: El historial del paciente.
2. `ADMISSIONS`: Su registro de ingreso.
3. `ICUSTAYS`: Su registro de estancia en Cuidados Intensivos (UCI).
4. `DIAGNOSES_ICD`: Su código de diagnóstico médico de la enfermedad.

**¿Qué datos usaremos al final?**
Pasaremos de archivos fragmentados a un Súper-Dataset consolidado que tendrá **1897 filas unidas** y **23 columnas combinadas** como: `gender`, `admission_type` y `icd9_code` (Código de diagnóstico).  

Aquí procesaremos y prepararemos estos datos recién unidos matemáticamente desde cero usando fórmulas puras, tal como se hace en el curso. Explicaremos paso a paso el código de la unión.

## 2. Importación de Librerías
Vamos a cargar las herramientas necesarias, usando lo más básico posible:

- `pandas` y `numpy`: para leer el archivo y manipular la tabla matemáticamente.
- `matplotlib.pyplot` y `seaborn`: para hacer gráficos simples en 2D (líneas, barras), nada en 3D para no complicarlo.

*Nota:* No usaremos funciones complejas o cajas negras para normalizar ni para dividir los datos, todo lo haremos "a mano" con simples matemáticas.


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 3. Cargar el Archivo (Dataset) y realizar Uniones (Merge) Múltiples
En el área de la salud, la información de un paciente viene fragmentada. 

Para demostrar un caso completo de Machine Learning, en lugar de usar solo dos archivos, **uniremos 4 tablas principales** de este dataset hospitalario:
1. `PATIENTS.csv`
2. `ADMISSIONS.csv`
3. `ICUSTAYS.csv` (Cuidados intensivos)
4. `DIAGNOSES_ICD.csv` (Diagnóstico)

Las uniremos progresivamente usando las llaves primarias en común que tiene el hospital (`subject_id` para el paciente y `hadm_id` para la admisión actual). Luego, descartaremos las puras fechas exactas porque sin transformaciones complejas de tiempo, arruinarían el modelo matemático.

In [13]:
# Ruta a la carpeta que contiene múltiples archivos CSV
ruta_base = 'REPOSITORIOS_PARCIAL/2_MIMIC‑III (Medical Information Mart for Intensive Care)/mimic-iii-clinical-database-demo-1.4/'

# 1. Leemos 4 tablas principales
pacientes = pd.read_csv(ruta_base + 'PATIENTS.csv')
admisiones = pd.read_csv(ruta_base + 'ADMISSIONS.csv')
uci_estancia = pd.read_csv(ruta_base + 'ICUSTAYS.csv')
diagnosticos = pd.read_csv(ruta_base + 'DIAGNOSES_ICD.csv')

# Previo a la unión, podemos borrar la columna repetida e inútil 'row_id' de todos
# para no tener errores de duplicación con las variables x e y
pacientes = pacientes.drop(columns=['row_id'])
admisiones = admisiones.drop(columns=['row_id'])
uci_estancia = uci_estancia.drop(columns=['row_id'])
diagnosticos = diagnosticos.drop(columns=['row_id'])

# 2. Las unimos (Merge) secuencialmente en un solo super-dataset
# a) Pacientes + Admisiones usando "subject_id"
df1 = pd.merge(pacientes, admisiones, on='subject_id')

# b) El resultado anterior + UCI Estancia usando "subject_id" y además "hadm_id" (Código de la visita)
df2 = pd.merge(df1, uci_estancia, on=['subject_id', 'hadm_id'])

# c) El gran resultado + Diagnóstico de esa Estancia
tabla_datos = pd.merge(df2, diagnosticos, on=['subject_id', 'hadm_id'])

# 3. Opcional: Descartamos momentáneamente fechas exactas ('2152-09-12 10:00:00')
# porque al convertir todo a matemática generarían miles de columnas basura sin aportar nada a la lógica base:
columnas_fechas_y_secuencias = [
    'seq_num', 'dob', 'dod', 'dod_hosp', 'dod_ssn', 'admittime', 
    'dischtime', 'deathtime', 'edregtime', 'edouttime', 
    'intime', 'outtime'
]
tabla_datos = tabla_datos.drop(columns=columnas_fechas_y_secuencias, errors='ignore')

# 4. Mostramos tamaño (filas, columnas) de nuestro nuevo Súper Dataset de 4 tablas unidas:
print('Tamaño del súper-dataset unido final:', tabla_datos.shape)

# Vemos las primeras 5 filas combinadas
tabla_datos.head()
# Imprimir información detallada del dataset
filas, columnas = tabla_datos.shape
print(f"\n--- INFORMACIÓN DEL DATASET ---")
print(f"Total de Filas (Ejemplos): {filas}")
print(f"Total de Columnas (Características): {columnas}\n")


Tamaño del súper-dataset unido final: (1897, 23)


,subject_id,gender,expire_flag,hadm_id,admission_type,admission_location,discharge_location,insurance,language,religion,...,hospital_expire_flag,has_chartevents_data,icustay_id,dbsource,first_careunit,last_careunit,first_wardid,last_wardid,los,icd9_code
0,10006,F,1,142345,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,...,0,1,206504,carevue,MICU,MICU,52,52,1.6325,99591
1,10006,F,1,142345,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,...,0,1,206504,carevue,MICU,MICU,52,52,1.6325,99662
2,10006,F,1,142345,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,...,0,1,206504,carevue,MICU,MICU,52,52,1.6325,5672
3,10006,F,1,142345,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,...,0,1,206504,carevue,MICU,MICU,52,52,1.6325,40391
4,10006,F,1,142345,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,...,0,1,206504,carevue,MICU,MICU,52,52,1.6325,42731


## 4. Análisis de Datos Vacíos o Faltantes
En la vida real, los datos no siempre vienen perfectos, hay casillas vacías (Nulos o NaN). 

En este caso, hemos decidido **eliminar la fila completa** si detectamos que tiene algún dato faltante. Esto asegura que la red neuronal solo aprenda de casos reales y 100% verídicos (no inventamos promedios ni asumimos valores vacíos), a costa de reducir la cantidad de ejemplos disponibles.


In [14]:
datos_nulos = tabla_datos.isnull().sum()
print('Columnas con casillas vacías antes de limpieza:')
print(datos_nulos[datos_nulos > 0])

# Eliminamos absolutamente cualquier fila que contenga un dato nulo (NaN)
filas_antes = len(tabla_datos)
tabla_datos = tabla_datos.dropna()
filas_despues = len(tabla_datos)

print(f"\nSe eliminaron {filas_antes - filas_despues} filas que contenían datos nulos.")
print('Nulos restantes sumados:', tabla_datos.isnull().sum().sum())


Columnas con casillas vacías:
language          527
religion            9
marital_status    143
dtype: int64

Nulos restantes sumados: 0


C:\Users\alfao\AppData\Local\Temp\ipykernel_14220\885227427.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = tabla_datos.select_dtypes(include=['object']).columns


## 5. Conversión de Textos a Números
Como la computadora o nuestra Red Neuronal no entiende palabras , necesitamos números. Usaremos una herramienta integrada muy sencilla llamada `get_dummies`.

Lo que hace es tomar las columnas de texto y crear columnas visuales llenas de ceros (`0`) y unos (`1`). Es directo y no necesitas complicarlo con diccionarios o For loops.


In [15]:
# Convertimos todo el texto a datos de 0 y 1 de forma directa y sencilla.
tabla_datos = pd.get_dummies(tabla_datos, dtype=int)

# Mostramos cómo quedó la tabla ahora, lista solo con formato numérico
tabla_datos.head()


,subject_id,expire_flag,hadm_id,hospital_expire_flag,has_chartevents_data,icustay_id,first_wardid,last_wardid,los,gender_F,...,icd9_code_V5419,icd9_code_V550,icd9_code_V551,icd9_code_V5861,icd9_code_V5865,icd9_code_V5867,icd9_code_V600,icd9_code_V667,icd9_code_V8741,icd9_code_V8801
0,10006,1,142345,0,1,206504,52,52,1.6325,1,...,0,0,0,0,0,0,0,0,0,0
1,10006,1,142345,0,1,206504,52,52,1.6325,1,...,0,0,0,0,0,0,0,0,0,0
2,10006,1,142345,0,1,206504,52,52,1.6325,1,...,0,0,0,0,0,0,0,0,0,0
3,10006,1,142345,0,1,206504,52,52,1.6325,1,...,0,0,0,0,0,0,0,0,0,0
4,10006,1,142345,0,1,206504,52,52,1.6325,1,...,0,0,0,0,0,0,0,0,0,0


## 6. Normalización de características (Feature Normalize)
Al visualizar los datos se puede observar que las características tienen diferentes magnitudes (por ejemplo, precios altísimos y números pequeños).

**La fórmula para calcular esto con escalado Z-Score es:**
$$ X_{norm} = \frac{X - \mu}{\sigma} $$

En código lo aplicamos usando las operaciones directas `mean()` (media) y `std()` (desviación estándar).


In [16]:
# 1. Calculamos la media (mu) y desviación estándar (sigma) para cada una de las columnas
mu = tabla_datos.mean()
sigma = tabla_datos.std()

# Para evitar errores por dividir entre cero (si la desviación es 0 en alguna columna), forzamos que sea mínimo 1
sigma_seguro = np.where(sigma == 0, 1, sigma)

# 2. Aplicamos la fórmula matemática exacta: (X - mu) / sigma
datos_normalizados = (tabla_datos - mu) / sigma_seguro

# Mostrar la tabla final convertida
datos_normalizados.head()


,subject_id,expire_flag,hadm_id,hospital_expire_flag,has_chartevents_data,icustay_id,first_wardid,last_wardid,los,gender_F,...,icd9_code_V5419,icd9_code_V550,icd9_code_V551,icd9_code_V5861,icd9_code_V5865,icd9_code_V5867,icd9_code_V600,icd9_code_V667,icd9_code_V8741,icd9_code_V8801
0,-1.339359,0.0,-0.407406,-0.709997,0.083046,-1.535589,0.905282,0.885739,-0.503035,1.115488,...,-0.02296,-0.02296,-0.02296,-0.100558,-0.02296,-0.06506,-0.02296,-0.039788,-0.039788,-0.039788
1,-1.339359,0.0,-0.407406,-0.709997,0.083046,-1.535589,0.905282,0.885739,-0.503035,1.115488,...,-0.02296,-0.02296,-0.02296,-0.100558,-0.02296,-0.06506,-0.02296,-0.039788,-0.039788,-0.039788
2,-1.339359,0.0,-0.407406,-0.709997,0.083046,-1.535589,0.905282,0.885739,-0.503035,1.115488,...,-0.02296,-0.02296,-0.02296,-0.100558,-0.02296,-0.06506,-0.02296,-0.039788,-0.039788,-0.039788
3,-1.339359,0.0,-0.407406,-0.709997,0.083046,-1.535589,0.905282,0.885739,-0.503035,1.115488,...,-0.02296,-0.02296,-0.02296,-0.100558,-0.02296,-0.06506,-0.02296,-0.039788,-0.039788,-0.039788
4,-1.339359,0.0,-0.407406,-0.709997,0.083046,-1.535589,0.905282,0.885739,-0.503035,1.115488,...,-0.02296,-0.02296,-0.02296,-0.100558,-0.02296,-0.06506,-0.02296,-0.039788,-0.039788,-0.039788


## 7. Dividir el Dataset en 80/20 (Entrenamiento y Prueba)
Finalmente apartaremos el 80% de nuestros datos para que el modelo aprenda y un 20% para hacerle un examen y evaluarlo.
En lugar de una caja negra, apartamos las filas simplemente cortando las tablas basándonos en el porcentaje.


In [17]:
# Extraemos la última columna para usarla como objetivo de predicción (Variable Y o Target)
objetivo_col = datos_normalizados.columns[-1]
caracteristicas = datos_normalizados.drop(objetivo_col, axis=1)
objetivo = datos_normalizados[objetivo_col]

# --------- DIVISIÓN A MANO (80% Entrenamiento, 20% Prueba) ---------

# 1. Calculamos cuánto es el 80% entero de nuestras filas
total_filas = len(datos_normalizados)
ochenta_por_ciento = int(total_filas * 0.8)

# 2. Cortamos las tablas usando posiciones desde 0 hasta el 80%
X_entrenamiento = caracteristicas.iloc[:ochenta_por_ciento]
y_entrenamiento = objetivo.iloc[:ochenta_por_ciento]

# 3. Y para el resto (el 20% restante), usamos desde el 80% en adelante
X_prueba = caracteristicas.iloc[ochenta_por_ciento:]
y_prueba = objetivo.iloc[ochenta_por_ciento:]

print('Total datos Original:', total_filas, 'filas')
print('Datos para Entrenar (80%):', len(X_entrenamiento), 'filas')
print('Datos para Probar (20%):', len(X_prueba), 'filas')
print('\n¡El Dataset está preparado puramente usando lógica matemática!')


Total datos Original: 1897 filas
Datos para Entrenar (80%): 1517 filas
Datos para Probar (20%): 380 filas

¡El Dataset está preparado puramente usando lógica matemática!
